# 1. Data Processing

## Importing all the nesessary Libraries

In [10]:
# importing all the Necessary Libraries

import numpy as np
import pandas as pd
from pathlib import Path
import os


##  Uploading the Dataset and Merging into 1 Dataset

In [15]:
# Create an empty list to merge all the datasets
all_patients_list = []
#Load your list of ID's from the csv file
patient_ids = pd.read_csv("T1DM_patient_sleep_demographics_with_race.csv")['Patient_ID'].tolist()

# Slice the list to take only the first 5 Patient IDs
first5patients = patient_ids[:5]

# Loading file path to access the files
patientfiles = Path("HUPA-UC Diabetes Dataset1-5")

patient_files = sorted(patientfiles.glob("*.csv"))

#Loop for processing all the files and merging into 1 file 
for patient_files in first5patients:
    df = pd.read_csv(patientfiles, sep = ";", index_col = None, header= 0)

    # Creating a new column to add Patient ID's
    df['Patient_ID'] = patient_files.stem
    all_patients_list.append(df)

#------------MERGING EVERYTHING-------------
cleaned_data = pd.concat (all_patients_list, ignore_index = True)
        
    ##----------------SAVE TE CLEANED FILE----------------------------
cleaned_data.to_csv("cleaned_file.csv", index = False)


   

FileNotFoundError: [Errno 2] No such file or directory: 'T1DM_patient_sleep_demographics_with_race.csv'

## Cleaning Steps

1. Add Patient ID column
2. Round numeric columns
3. Check the Datatypes
4. Convert Datatypes
5. Check the Outliers
    1. Glucose
    2. Calories
    3. Heart Rate

## Add Patient ID column to all the patient information files

In [85]:
def patientid(df,p_id):
    #Add Patient ID Column to the Patient information csv files
    # here we are paasing p_id (patient id value into the function so it know what to add
    df['patient_id'] = p_id
    return df

## Rounding all the numeric column values 

In [86]:
def rounding_numeric(df):
    # 1. Round numeric columns
    df = df.round(3)
    
    return df

## Check the all column datatypes and change if necessary

In [92]:
def  fix_Dataypes(df): # 2. Convert time (Fixed the 'df1' typo here)
    df.dtypes
    df['time'] = pd.to_datetime(df['time'])
    
    return df
    

## Glucose Outliers

trying to remove biologically impoosible or indicating a sensor failure:\
**40mg/dl** is near the limit of survial; \
**500mg/dl** is a critical emergency.

In [93]:
def glucose_outliers(df):
        #remove impossible values for a living human
    # .Copy() without copy, if you delete the orinal big dataset, the filtered one might still be secretly tethered to it in memory.
    # but with adding copy() we can have a clean break. the filtered df stands on its own.
    df = df[(df['glucose']>= 40) & (df['glucose'] <= 500)].copy()
     # 6. Reset index
    df = df.reset_index(drop=True)
    return df

## Heart Rate Outliers

**Heart rate range**
**30-220 bpm** is a standard, resonable range for filtering impossible biological data in humans.

In [94]:
 def heart_rate_outliers(df): # Remove Heart Rate Outliers
    if 'heart_rate' in df.columns:
        # Remove impossible biological values 
        df = df[(df['heart_rate']>=30 ) & (df['heart_rate'] <= 220)].copy()
         # 6. Reset index
        df = df.reset_index(drop=True)
        return df